# 🔍 Optimización — S7## Federated Proactive Forest**Estrategia:** Ranks trees by per-client combined F1 + PCD score.**Hiperparámetros:** `alpha_pf`, `t_max`, `f1_weight`, `local_weight`**Datasets:** Letter, Optdigits, Spambase, Nursery, Sonar, Vowel**N_CLIENTS:** 3> ⚡ **Cada celda de dataset es independiente** — ejecuta solo la que necesites.> 📋 **Rangos leídos en runtime desde** `configs/experiments/optimization/search_spaces.yaml`

In [ ]:
# ── Imports & Config ─────────────────────────────────────────────────────import sysfrom pathlib import PathROOT = Path.cwd().parent.parent.parent.parentsys.path.insert(0, str(ROOT))import numpy as npimport pandas as pdimport yamlimport jsonimport optunafrom sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScalerfrom sklearn.model_selection import train_test_splitSEED = 42N_CLIENTS = 3N_TRIALS = 20STRATEGY = 'S7'DATA_DIR = ROOT / 'data'RESULTS_DIR = ROOT / 'results' / 's6_alpha_optimization'RESULTS_DIR.mkdir(parents=True, exist_ok=True)# Load search space from YAML (single source of truth)with open(ROOT / 'configs' / 'experiments' / 'optimization' / 'search_spaces.yaml') as f:    all_spaces = yaml.safe_load(f)SPACE = all_spaces[STRATEGY]from src.domain.dataset.base_adapter import DatasetSplitfrom src.application.fl_orchestrator import FLEXOrchestratorprint(f'✅ Project root: {ROOT}')print(f'✅ Strategy: {STRATEGY}')print(f'✅ N_CLIENTS: {N_CLIENTS}')print(f'✅ Search space: {list(SPACE.keys())}')

## Dataset: Letter20,000 samples, 16 features, 26 classes (A-Z), numeric features

In [ ]:
# ── Load Letter ──────────────────────────────────────────────────df = pd.read_csv(DATA_DIR / 'letter.csv')print(f'📊 Shape: {df.shape}')X = df.drop(columns=['class']).values.astype(float)le = LabelEncoder()y = le.fit_transform(df['class'])class_names = list(le.classes_)feature_names = [c for c in df.columns if c != 'class']X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)sc = StandardScaler()X_tr = sc.fit_transform(X_tr)X_te = sc.transform(X_te)ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,                  feature_names=feature_names, class_names=class_names,                  dataset_name='letter')print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')# ── Optimization ──────────────────────────────────────────────────────────def objective(trial):    params = {}    for _name, _cfg in SPACE.items():        _t = _cfg.get('type', 'float')        _low, _high = _cfg['low'], _cfg['high']        _step = _cfg.get('step')        _log = _cfg.get('log', False)        if _t == 'float':            if _log:                params[_name] = trial.suggest_float(_name, _low, _high, log=True)            elif _step is not None:                params[_name] = trial.suggest_float(_name, _low, _high, step=_step)            else:                params[_name] = trial.suggest_float(_name, _low, _high)        elif _t == 'int':            params[_name] = trial.suggest_int(_name, _low, _high, step=_step or 1)    return params        agg = {'strategy': STRATEGY}        if 't_max' in params:            agg['t_max'] = params['t_max']        if 'f1_weight' in params:            agg['f1_weight'] = params['f1_weight']            agg['pcd_weight'] = 1.0 - params['f1_weight']        cfg = {            'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},            'model': {                'n_estimators': 100, 'alpha': params['alpha_pf'],                'split_criterion': 'entropy', 'use_progressive_stopping': True,                'convergence': 0.002, 'episode_size': 5, 'verbose': False,            },            'aggregation': agg,            'prediction': {                'local_weight': params['local_weight'],                'global_weight': 1.0 - params['local_weight'],            },            'verbose': False, 'seed': SEED,        }        np.random.seed(SEED)        orch = FLEXOrchestrator.from_config(cfg)        orch.setup_federation(ds, seed=SEED)        res = orch.run_federated_round()        return res.global_macro_f1print(f"🚀 Optimizando S7 en {ds.dataset_name}... ({N_TRIALS} trials)")study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)df_trials = study.trials_dataframe()print(f"\n{'='*70}")print(f"📊 S7 — {ds.dataset_name} — Resultados")print(f"{'='*70}")print(f"   Mejor Macro-F1: {study.best_value:.4f}")for p, v in study.best_params.items():    print(f"   {p:25s}: {v}")print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")# Save per-dataset resultsresult = {    'dataset': 'letter',    'strategy': 'S7',    'best_macro_f1': study.best_value,    'mean_macro_f1': df_trials['value'].mean(),    'std_macro_f1': df_trials['value'].std(),    'best_params': study.best_params,    'all_trials': df_trials.to_dict('records'),}with open(RESULTS_DIR / 's6_letter_s7_results.json', 'w') as f:    json.dump(result, f, indent=2, default=str)print(f"\n✅ Resultados guardados: s6_letter_s7_results.json")

## Dataset: Optdigits5,620 samples, 64 features (8x8 pixel), 10 classes (0-9), numeric 0-16

In [ ]:
# ── Load Optdigits ──────────────────────────────────────────────────df = pd.read_csv(DATA_DIR / 'optdigits.csv')print(f'📊 Shape: {df.shape}')X = df.drop(columns=['class']).values.astype(float)le = LabelEncoder()y = le.fit_transform(df['class'])class_names = list(le.classes_)feature_names = [c for c in df.columns if c != 'class']X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)sc = StandardScaler()X_tr = sc.fit_transform(X_tr)X_te = sc.transform(X_te)ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,                  feature_names=feature_names, class_names=class_names,                  dataset_name='optdigits')print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')# ── Optimization ──────────────────────────────────────────────────────────def objective(trial):    params = {}    for _name, _cfg in SPACE.items():        _t = _cfg.get('type', 'float')        _low, _high = _cfg['low'], _cfg['high']        _step = _cfg.get('step')        _log = _cfg.get('log', False)        if _t == 'float':            if _log:                params[_name] = trial.suggest_float(_name, _low, _high, log=True)            elif _step is not None:                params[_name] = trial.suggest_float(_name, _low, _high, step=_step)            else:                params[_name] = trial.suggest_float(_name, _low, _high)        elif _t == 'int':            params[_name] = trial.suggest_int(_name, _low, _high, step=_step or 1)    return params        agg = {'strategy': STRATEGY}        if 't_max' in params:            agg['t_max'] = params['t_max']        if 'f1_weight' in params:            agg['f1_weight'] = params['f1_weight']            agg['pcd_weight'] = 1.0 - params['f1_weight']        cfg = {            'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},            'model': {                'n_estimators': 100, 'alpha': params['alpha_pf'],                'split_criterion': 'entropy', 'use_progressive_stopping': True,                'convergence': 0.002, 'episode_size': 5, 'verbose': False,            },            'aggregation': agg,            'prediction': {                'local_weight': params['local_weight'],                'global_weight': 1.0 - params['local_weight'],            },            'verbose': False, 'seed': SEED,        }        np.random.seed(SEED)        orch = FLEXOrchestrator.from_config(cfg)        orch.setup_federation(ds, seed=SEED)        res = orch.run_federated_round()        return res.global_macro_f1print(f"🚀 Optimizando S7 en {ds.dataset_name}... ({N_TRIALS} trials)")study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)df_trials = study.trials_dataframe()print(f"\n{'='*70}")print(f"📊 S7 — {ds.dataset_name} — Resultados")print(f"{'='*70}")print(f"   Mejor Macro-F1: {study.best_value:.4f}")for p, v in study.best_params.items():    print(f"   {p:25s}: {v}")print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")# Save per-dataset resultsresult = {    'dataset': 'optdigits',    'strategy': 'S7',    'best_macro_f1': study.best_value,    'mean_macro_f1': df_trials['value'].mean(),    'std_macro_f1': df_trials['value'].std(),    'best_params': study.best_params,    'all_trials': df_trials.to_dict('records'),}with open(RESULTS_DIR / 's6_optdigits_s7_results.json', 'w') as f:    json.dump(result, f, indent=2, default=str)print(f"\n✅ Resultados guardados: s6_optdigits_s7_results.json")

## Dataset: Spambase4,601 samples, 57 features (word frequencies), 2 classes (spam/ham)

In [ ]:
# ── Load Spambase ──────────────────────────────────────────────────df = pd.read_csv(DATA_DIR / 'spambase.csv')print(f'📊 Shape: {df.shape}')X = df.drop(columns=['class']).values.astype(float)le = LabelEncoder()y = le.fit_transform(df['class'])class_names = list(le.classes_)feature_names = [c for c in df.columns if c != 'class']X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)sc = StandardScaler()X_tr = sc.fit_transform(X_tr)X_te = sc.transform(X_te)ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,                  feature_names=feature_names, class_names=class_names,                  dataset_name='spambase')print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')# ── Optimization ──────────────────────────────────────────────────────────def objective(trial):    params = {}    for _name, _cfg in SPACE.items():        _t = _cfg.get('type', 'float')        _low, _high = _cfg['low'], _cfg['high']        _step = _cfg.get('step')        _log = _cfg.get('log', False)        if _t == 'float':            if _log:                params[_name] = trial.suggest_float(_name, _low, _high, log=True)            elif _step is not None:                params[_name] = trial.suggest_float(_name, _low, _high, step=_step)            else:                params[_name] = trial.suggest_float(_name, _low, _high)        elif _t == 'int':            params[_name] = trial.suggest_int(_name, _low, _high, step=_step or 1)    return params        agg = {'strategy': STRATEGY}        if 't_max' in params:            agg['t_max'] = params['t_max']        if 'f1_weight' in params:            agg['f1_weight'] = params['f1_weight']            agg['pcd_weight'] = 1.0 - params['f1_weight']        cfg = {            'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},            'model': {                'n_estimators': 100, 'alpha': params['alpha_pf'],                'split_criterion': 'entropy', 'use_progressive_stopping': True,                'convergence': 0.002, 'episode_size': 5, 'verbose': False,            },            'aggregation': agg,            'prediction': {                'local_weight': params['local_weight'],                'global_weight': 1.0 - params['local_weight'],            },            'verbose': False, 'seed': SEED,        }        np.random.seed(SEED)        orch = FLEXOrchestrator.from_config(cfg)        orch.setup_federation(ds, seed=SEED)        res = orch.run_federated_round()        return res.global_macro_f1print(f"🚀 Optimizando S7 en {ds.dataset_name}... ({N_TRIALS} trials)")study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)df_trials = study.trials_dataframe()print(f"\n{'='*70}")print(f"📊 S7 — {ds.dataset_name} — Resultados")print(f"{'='*70}")print(f"   Mejor Macro-F1: {study.best_value:.4f}")for p, v in study.best_params.items():    print(f"   {p:25s}: {v}")print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")# Save per-dataset resultsresult = {    'dataset': 'spambase',    'strategy': 'S7',    'best_macro_f1': study.best_value,    'mean_macro_f1': df_trials['value'].mean(),    'std_macro_f1': df_trials['value'].std(),    'best_params': study.best_params,    'all_trials': df_trials.to_dict('records'),}with open(RESULTS_DIR / 's6_spambase_s7_results.json', 'w') as f:    json.dump(result, f, indent=2, default=str)print(f"\n✅ Resultados guardados: s6_spambase_s7_results.json")

## Dataset: Nursery12,960 samples, 8 categorical features, 5 classes

In [ ]:
# ── Load Nursery ──────────────────────────────────────────────────df = pd.read_csv(DATA_DIR / 'nursery.csv')print(f'📊 Shape: {df.shape}')cat_cols = ['parents', 'has_nurs', 'form', 'children', 'housing', 'finance', 'social', 'health']enc = OrdinalEncoder()X = enc.fit_transform(df[cat_cols])le = LabelEncoder()y = le.fit_transform(df['class'])class_names = list(le.classes_)feature_names = [c for c in df.columns if c != 'class']X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)sc = StandardScaler()X_tr = sc.fit_transform(X_tr)X_te = sc.transform(X_te)ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,                  feature_names=feature_names, class_names=class_names,                  dataset_name='nursery')print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')# ── Optimization ──────────────────────────────────────────────────────────def objective(trial):    params = {}    for _name, _cfg in SPACE.items():        _t = _cfg.get('type', 'float')        _low, _high = _cfg['low'], _cfg['high']        _step = _cfg.get('step')        _log = _cfg.get('log', False)        if _t == 'float':            if _log:                params[_name] = trial.suggest_float(_name, _low, _high, log=True)            elif _step is not None:                params[_name] = trial.suggest_float(_name, _low, _high, step=_step)            else:                params[_name] = trial.suggest_float(_name, _low, _high)        elif _t == 'int':            params[_name] = trial.suggest_int(_name, _low, _high, step=_step or 1)    return params        agg = {'strategy': STRATEGY}        if 't_max' in params:            agg['t_max'] = params['t_max']        if 'f1_weight' in params:            agg['f1_weight'] = params['f1_weight']            agg['pcd_weight'] = 1.0 - params['f1_weight']        cfg = {            'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},            'model': {                'n_estimators': 100, 'alpha': params['alpha_pf'],                'split_criterion': 'entropy', 'use_progressive_stopping': True,                'convergence': 0.002, 'episode_size': 5, 'verbose': False,            },            'aggregation': agg,            'prediction': {                'local_weight': params['local_weight'],                'global_weight': 1.0 - params['local_weight'],            },            'verbose': False, 'seed': SEED,        }        np.random.seed(SEED)        orch = FLEXOrchestrator.from_config(cfg)        orch.setup_federation(ds, seed=SEED)        res = orch.run_federated_round()        return res.global_macro_f1print(f"🚀 Optimizando S7 en {ds.dataset_name}... ({N_TRIALS} trials)")study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)df_trials = study.trials_dataframe()print(f"\n{'='*70}")print(f"📊 S7 — {ds.dataset_name} — Resultados")print(f"{'='*70}")print(f"   Mejor Macro-F1: {study.best_value:.4f}")for p, v in study.best_params.items():    print(f"   {p:25s}: {v}")print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")# Save per-dataset resultsresult = {    'dataset': 'nursery',    'strategy': 'S7',    'best_macro_f1': study.best_value,    'mean_macro_f1': df_trials['value'].mean(),    'std_macro_f1': df_trials['value'].std(),    'best_params': study.best_params,    'all_trials': df_trials.to_dict('records'),}with open(RESULTS_DIR / 's6_nursery_s7_results.json', 'w') as f:    json.dump(result, f, indent=2, default=str)print(f"\n✅ Resultados guardados: s6_nursery_s7_results.json")

## Dataset: Sonar208 samples, 60 numeric features, 2 classes (Rock/Mine) — small dataset!

In [ ]:
# ── Load Sonar ──────────────────────────────────────────────────df = pd.read_csv(DATA_DIR / 'sonar.csv')print(f'📊 Shape: {df.shape}')X = df.drop(columns=['Class']).values.astype(float)le = LabelEncoder()y = le.fit_transform(df['Class'])class_names = list(le.classes_)feature_names = [c for c in df.columns if c != 'Class']X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)sc = StandardScaler()X_tr = sc.fit_transform(X_tr)X_te = sc.transform(X_te)ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,                  feature_names=feature_names, class_names=class_names,                  dataset_name='sonar')print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')# ── Optimization ──────────────────────────────────────────────────────────def objective(trial):    params = {}    for _name, _cfg in SPACE.items():        _t = _cfg.get('type', 'float')        _low, _high = _cfg['low'], _cfg['high']        _step = _cfg.get('step')        _log = _cfg.get('log', False)        if _t == 'float':            if _log:                params[_name] = trial.suggest_float(_name, _low, _high, log=True)            elif _step is not None:                params[_name] = trial.suggest_float(_name, _low, _high, step=_step)            else:                params[_name] = trial.suggest_float(_name, _low, _high)        elif _t == 'int':            params[_name] = trial.suggest_int(_name, _low, _high, step=_step or 1)    return params        agg = {'strategy': STRATEGY}        if 't_max' in params:            agg['t_max'] = params['t_max']        if 'f1_weight' in params:            agg['f1_weight'] = params['f1_weight']            agg['pcd_weight'] = 1.0 - params['f1_weight']        cfg = {            'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},            'model': {                'n_estimators': 100, 'alpha': params['alpha_pf'],                'split_criterion': 'entropy', 'use_progressive_stopping': True,                'convergence': 0.002, 'episode_size': 5, 'verbose': False,            },            'aggregation': agg,            'prediction': {                'local_weight': params['local_weight'],                'global_weight': 1.0 - params['local_weight'],            },            'verbose': False, 'seed': SEED,        }        np.random.seed(SEED)        orch = FLEXOrchestrator.from_config(cfg)        orch.setup_federation(ds, seed=SEED)        res = orch.run_federated_round()        return res.global_macro_f1print(f"🚀 Optimizando S7 en {ds.dataset_name}... ({N_TRIALS} trials)")study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)df_trials = study.trials_dataframe()print(f"\n{'='*70}")print(f"📊 S7 — {ds.dataset_name} — Resultados")print(f"{'='*70}")print(f"   Mejor Macro-F1: {study.best_value:.4f}")for p, v in study.best_params.items():    print(f"   {p:25s}: {v}")print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")# Save per-dataset resultsresult = {    'dataset': 'sonar',    'strategy': 'S7',    'best_macro_f1': study.best_value,    'mean_macro_f1': df_trials['value'].mean(),    'std_macro_f1': df_trials['value'].std(),    'best_params': study.best_params,    'all_trials': df_trials.to_dict('records'),}with open(RESULTS_DIR / 's6_sonar_s7_results.json', 'w') as f:    json.dump(result, f, indent=2, default=str)print(f"\n✅ Resultados guardados: s6_sonar_s7_results.json")

## Dataset: Vowel990 samples, 10 numeric features (drop 3 metadata cols), 11 classes

In [ ]:
# ── Load Vowel ──────────────────────────────────────────────────df = pd.read_csv(DATA_DIR / 'vowel.csv')print(f'📊 Shape: {df.shape}')df = df.drop(columns=['Train or Test', 'Speaker Number', 'Sex'])X = df.drop(columns=['Class']).values.astype(float)le = LabelEncoder()y = le.fit_transform(df['Class'])class_names = list(le.classes_)feature_names = [c for c in df.columns if c != 'Class']X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)sc = StandardScaler()X_tr = sc.fit_transform(X_tr)X_te = sc.transform(X_te)ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,                  feature_names=feature_names, class_names=class_names,                  dataset_name='vowel')print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')# ── Optimization ──────────────────────────────────────────────────────────def objective(trial):    params = {}    for _name, _cfg in SPACE.items():        _t = _cfg.get('type', 'float')        _low, _high = _cfg['low'], _cfg['high']        _step = _cfg.get('step')        _log = _cfg.get('log', False)        if _t == 'float':            if _log:                params[_name] = trial.suggest_float(_name, _low, _high, log=True)            elif _step is not None:                params[_name] = trial.suggest_float(_name, _low, _high, step=_step)            else:                params[_name] = trial.suggest_float(_name, _low, _high)        elif _t == 'int':            params[_name] = trial.suggest_int(_name, _low, _high, step=_step or 1)    return params        agg = {'strategy': STRATEGY}        if 't_max' in params:            agg['t_max'] = params['t_max']        if 'f1_weight' in params:            agg['f1_weight'] = params['f1_weight']            agg['pcd_weight'] = 1.0 - params['f1_weight']        cfg = {            'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},            'model': {                'n_estimators': 100, 'alpha': params['alpha_pf'],                'split_criterion': 'entropy', 'use_progressive_stopping': True,                'convergence': 0.002, 'episode_size': 5, 'verbose': False,            },            'aggregation': agg,            'prediction': {                'local_weight': params['local_weight'],                'global_weight': 1.0 - params['local_weight'],            },            'verbose': False, 'seed': SEED,        }        np.random.seed(SEED)        orch = FLEXOrchestrator.from_config(cfg)        orch.setup_federation(ds, seed=SEED)        res = orch.run_federated_round()        return res.global_macro_f1print(f"🚀 Optimizando S7 en {ds.dataset_name}... ({N_TRIALS} trials)")study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)df_trials = study.trials_dataframe()print(f"\n{'='*70}")print(f"📊 S7 — {ds.dataset_name} — Resultados")print(f"{'='*70}")print(f"   Mejor Macro-F1: {study.best_value:.4f}")for p, v in study.best_params.items():    print(f"   {p:25s}: {v}")print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")# Save per-dataset resultsresult = {    'dataset': 'vowel',    'strategy': 'S7',    'best_macro_f1': study.best_value,    'mean_macro_f1': df_trials['value'].mean(),    'std_macro_f1': df_trials['value'].std(),    'best_params': study.best_params,    'all_trials': df_trials.to_dict('records'),}with open(RESULTS_DIR / 's6_vowel_s7_results.json', 'w') as f:    json.dump(result, f, indent=2, default=str)print(f"\n✅ Resultados guardados: s6_vowel_s7_results.json")

## 📊 Resumen Global — Comparar todos los datasets> ⚡ Ejecuta esta celda **después** de haber ejecutado todas las celdas de datasets.

In [ ]:
# ── Resumen Global (ejecutar después de todas las celdas) ─────────────results = []for fp in sorted(RESULTS_DIR.glob('s6_*_s7_results.json')):    with open(fp) as f:        r = json.load(f)    row = {'dataset': r['dataset'], 'best_f1': round(r['best_macro_f1'], 4),           'mean_f1': round(r['mean_macro_f1'], 4), 'std': round(r['std_macro_f1'], 4)}    row.update(r['best_params'])    results.append(row)df_sum = pd.DataFrame(results)print(f"\n{'='*90}")print(f"📋 RESUMEN GLOBAL — S7")print(f"{'='*90}")if df_sum.empty:    print('⚠️ No hay resultados. Ejecuta al menos una celda de dataset primero.')else:    print(df_sum.to_string(index=False))    if 'alpha_pf' in df_sum.columns:        rec = df_sum['alpha_pf'].median()        print(f"\n🎯 alpha_pf recomendado (mediana): {rec:.1f}")        print(f"   Rango: [{df_sum['alpha_pf'].min()} — {df_sum['alpha_pf'].max()}]")    df_sum.to_csv(RESULTS_DIR / f'summary_{strat_key.lower()}.csv', index=False)    print(f"\n✅ Resumen guardado: summary_{strat_key.lower()}.csv")